# Car Price Prediction - Feature Engineering

**Project:** Used Car Price Prediction  
**Course:** Data Science Capstone - 3rd Year  
**Author:** [Your Name]  
**Date:** 2024

In [ ]:
# Setup
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import config
from src.data.preprocessing import run_preprocessing_pipeline
from src.features.build_features import FeatureEngineer, build_features
from src.visualization.visualize import (
    plot_feature_importance,
    plot_correlation_heatmap,
    plot_target_vs_features,
    save_figure
)

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("Environment ready")

In [ ]:
# Load processed data
train_df, test_df = run_preprocessing_pipeline()

target_col = config.get('data.target_column', 'selling_price')
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

In [ ]:
# Build features
X_train, X_test, y_train, y_test = build_features(train_df, test_df)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nFeature columns ({len(X_train.columns)}):")
for i, col in enumerate(X_train.columns):
    print(f"  {i+1:2d}. {col}")

In [ ]:
# Inspect engineered features
engineered_cols = [c for c in X_train.columns if c not in [
    'year', 'km_driven', 'mileage', 'engine', 'max_power', 'seats',
    'fuel', 'seller_type', 'transmission', 'owner', 'brand'
]]

print(f"Engineered features: {engineered_cols}")

for col in engineered_cols:
    print(f"\n{col}:")
    print(f"  Train: mean={X_train[col].mean():.2f}, std={X_train[col].std():.2f}")
    print(f"  Test:  mean={X_test[col].mean():.2f}, std={X_test[col].std():.2f}")

In [ ]:
# Correlation with target (on engineered features)
train_with_target = X_train.copy()
train_with_target[target_col] = y_train

fig = plot_correlation_heatmap(train_with_target, target_col=target_col, save=True)
plt.show()

In [ ]:
# Target vs key engineered features
key_features = ['car_age', 'km_per_year', 'power_per_cc']
available = [f for f in key_features if f in X_train.columns]

if available:
    fig = plot_target_vs_features(train_with_target, target_col=target_col, features=available, save=True)
    plt.show()

In [ ]:
# Check categorical encoding
cat_cols = ['fuel', 'seller_type', 'transmission', 'owner', 'brand']
encoded_cols = [c for c in X_train.columns if any(c.startswith(cat) for cat in cat_cols)]

print(f"Encoded categorical columns ({len(encoded_cols)}):")
for col in encoded_cols:
    n_unique = X_train[col].nunique()
    print(f"  {col}: {n_unique} unique values")

In [ ]:
# Save feature engineer for later use
from src.features.build_features import save_feature_engineer

# Re-fit to get the fitted feature engineer
fe = FeatureEngineer()
fe.fit(X_train, y_train)
save_feature_engineer(fe)

print("Feature engineer saved to models/feature_engineer.joblib")

## Feature Engineering Summary

### Created Features:
1. **car_age**: Current year - manufacturing year (capped at 30)
2. **km_per_year**: km_driven / car_age (normalizes usage)
3. **power_per_cc**: max_power / (engine/1000) (performance proxy)
4. **mileage_category**: Quartile bins of mileage
5. **age_category**: Binned car age (new/recent/used/old)

### Encoding Strategy:
- Low cardinality (fuel, seller_type, transmission, owner): Label Encoding
- High cardinality (brand): Target Encoding (prevents overfitting)
- Dropped: name, torque

### Next Steps:
- Model training and comparison (notebook 03)
- Hyperparameter optimization
- Model evaluation and selection